# 01 Load Data

This notebook handles the initial data loading process for the Adult dataset.

## Assumptions & Data Quirks

- **Column Names**: The raw data files do not contain headers. We use the attribute list provided in the dataset description (age, workclass, fnlwgt, education, education-num, marital-status, occupation, relationship, race, sex, capital-gain, capital-loss, hours-per-week, native-country, income).
- **Missing Values**: Missing values are encoded as '?' in the raw files. These are handled by setting `na_values='?'` and `skipinitialspace=True` because the data is comma-space separated.
- **Test Set Header**: The `adult.test` file contains an extra header line (`|1x3 Cross validator`) which must be skipped.
- **Label Differences**: Labels in `adult.data` (e.g., `<=50K`) and `adult.test` (e.g., `<=50K.`) have different formats (trailing period in test set). This will need to be addressed in the cleaning phase.

In [5]:
import pandas as pd
import os

# Column names based on adult.names
COLUMNS = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num',
    'marital-status', 'occupation', 'relationship', 'race', 'sex',
    'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'
]

# Define paths
train_path = '../data/raw/adult.data'
test_path = '../data/raw/adult.test'

def load_dataset(path, skip_rows=0):
    return pd.read_csv(
        path,
        header=None,
        names=COLUMNS,
        skiprows=skip_rows,
        na_values='?',
        skipinitialspace=True
    )

# Load data
df_train = load_dataset(train_path)
df_test = load_dataset(test_path, skip_rows=1) # Skip the cross-validator header line

# Combine dataframes
df = pd.concat([df_train, df_test], axis=0, ignore_index=True)

print(f"Combined dataframe shape: {df.shape}")
df.head()

Combined dataframe shape: (48842, 15)


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [6]:
# 1) Print column names
print("Column Names:")
print(df.columns.tolist())
print("-" * 30)
# 2) Show data types
print("Data Types:")
print(df.dtypes)
print("-" * 30)
# 3) Count missing values per column
print("Missing Values per Column:")
print(df.isnull().sum())
print("-" * 30)
# 4) Show class distribution of the income target
print("Income Class Distribution (Normalized):")
print(df['income'].value_counts(normalize=True))
print("-" * 30)
# 5) Display summary statistics
print("Summary Statistics for Numerical Columns:")
display(df.describe())

Column Names:
['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']
------------------------------
Data Types:
age               int64
workclass           str
fnlwgt            int64
education           str
education-num     int64
marital-status      str
occupation          str
relationship        str
race                str
sex                 str
capital-gain      int64
capital-loss      int64
hours-per-week    int64
native-country      str
income              str
dtype: object
------------------------------
Missing Values per Column:
age                  0
workclass         2799
fnlwgt               0
education            0
education-num        0
marital-status       0
occupation        2809
relationship         0
race                 0
sex                  0
capital-gain         0
capital-loss         0
hours-per-week       0
native-cou

,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week
count,48842.000000,4.884200e+04,48842.000000,48842.000000,48842.000000,48842.000000
mean,38.643585,1.896641e+05,10.078089,1079.067626,87.502314,40.422382
std,13.710510,1.056040e+05,2.570973,7452.019058,403.004552,12.391444
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.175505e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.781445e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.376420e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.490400e+06,16.000000,99999.000000,4356.000000,99.000000


## Initial Data Observations ##
**Class Imbalance and Formatting:** The target variable income is imbalanced, with approximately 76% of entries earning <=50K and 24% earning >50K. The distribution also confirms that labels in the test set contain a trailing period (e.g., <=50K.), which accounts for ~33% of the total dataset and requires consolidation during cleaning.

**Scale of Missingness:** Missing values are concentrated in three categorical columns: occupation (2,809), workclass (2,799), and native-country (857). All other columns, including the target, are complete.

**Sparsity in Capital Metrics:** Summary statistics for capital-gain and capital-loss show that even at the 75th percentile, the values remain 0.0, indicating these features are highly sparse with extreme outliers (max gain of 99,999).

**Work Hours and Age Range:** The hours-per-week variable shows a standard work week (mean ~40.4) but contains a broad range from 1 to 99 hours. The age column spans from 17 to 90 years, with a mean of 38.6.

**Preprocessing Requirements:** The presence of object data types across 9 columns (e.g., workclass, education, marital-status) and the period-suffixed labels in income necessitate categorical encoding and string cleaning before modelling.

## Target Variable Cleaning ##

The labels in the test set (adult.test) contain a trailing period (e.g., <=50K.) that is not present in the training set. This correction is required to unify the labels into two distinct classes, ensuring the model treats them as the same categories across the combined dataset.

In [7]:
# Remove trailing periods from the income labels
df['income'] = df['income'].str.rstrip('.')
# Verify that only two classes remain
print(f"Unique income classes: {df['income'].unique().tolist()}")
# Recalculate and print updated class distribution
print("\nUpdated Income Class Distribution (Normalized):")
print(df['income'].value_counts(normalize=True))

Unique income classes: ['<=50K', '>50K']

Updated Income Class Distribution (Normalized):
income
<=50K    0.760718
>50K     0.239282
Name: proportion, dtype: float64


## Dropping Unused Features ##

The fnlwgt column represents a census sampling weight rather than a real predictive attribute of an individual. Including this weight in model training may distort the results by introducing bias from the data collection process.

In [8]:
# Drop the fnlwgt column
df.drop(columns=['fnlwgt'], inplace=True)
# Verify column removal
print(f"Remaining columns: {df.columns.tolist()}")

Remaining columns: ['age', 'workclass', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']
